# 01 – Auditoria dos dados

Dois caminhos: (A) planilha já agregada por mês (formato do LAPAE: uma linha por unidade ou segmento, uma coluna por mês) ou (B) registros brutos, um por exame. Sem nenhum dos dois em `dados/bruto/`, roda em **modo demonstração** (dados sintéticos); nenhum número desse modo é resultado do projeto.

In [1]:
from pathlib import Path
import sys
RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(RAIZ))
import pandas as pd
from src import config
from src.dados import carregar_bruto, carregar_mensal_agregado, limpar, agregar_mensal, validar_totais
from src.demo import gerar_registros_sinteticos, ROTULO_DEMO
pd.set_option("display.width", 140)
agregado = carregar_mensal_agregado()
print("Caminho A (planilha agregada por mês):", agregado is not None)

Caminho A (planilha agregada por mês): True


## Caminho A: planilha agregada
O código só lê `dados/bruto/`; nunca grava nele. Ausência de registro não é convertida em zero.

In [2]:
if agregado is not None:
    print("Arquivo:", agregado.attrs["arquivo_origem"])
    print("Linhas (unidade x mês):", len(agregado), "| unidades:", agregado[config.COL_UNIDADE].nunique(), "| meses:", agregado[config.COL_MES].nunique())
    print("Período:", f"{agregado[config.COL_MES].min():%Y-%m}", "a", f"{agregado[config.COL_MES].max():%Y-%m}")
    print("Valores ausentes:", int(agregado[config.COL_ALVO].isna().sum()), "| negativos:", int((agregado[config.COL_ALVO] < 0).sum()))
    # conferência: soma da base longa == soma da planilha original
    soma_longa = float(agregado[config.COL_ALVO].sum())
    assert soma_longa == agregado.attrs["total_planilha"], (soma_longa, agregado.attrs["total_planilha"])
    print("Total conferido com a planilha:", int(soma_longa))
    display(agregado.pivot(index=config.COL_UNIDADE, columns=config.COL_MES, values=config.COL_ALVO).rename(columns=lambda c: f"{c:%Y-%m}"))
    display(agregado.groupby(config.COL_MES)[config.COL_ALVO].sum().rename("total do mês").to_frame())

Arquivo: dados_lapae.xlsx
Linhas (unidade x mês): 84 | unidades: 6 | meses: 14
Período: 2025-08 a 2026-09
Valores ausentes: 0 | negativos: 0
Total conferido com a planilha: 20913


mes,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09
unidade,,,,,,,,,,,,,,
Maurílio de Almeida,262,602,728,417,324,528,433,573,417,437,462,593,543,494
Particulares,4,1,5,4,3,2,5,6,3,3,2,11,17,18
Roseane Dore,314,292,303,228,203,275,253,340,250,271,218,344,250,300
SUS,552,483,809,653,680,374,804,689,886,670,491,625,510,568
São Lucas,24,82,85,70,54,127,81,123,104,109,72,132,62,103
Viver Mais,9,10,41,20,9,5,6,3,2,8,6,15,15,4


,total do mês
mes,
2025-08-01,1165
2025-09-01,1470
2025-10-01,1971
2025-11-01,1392
2025-12-01,1273
2026-01-01,1311
2026-02-01,1582
2026-03-01,1734
2026-04-01,1662


In [3]:
if agregado is not None:
    agregado.to_csv(config.PROCESSADO / "base_mensal.csv", index=False)
    print("Gravado: dados/processado/base_mensal.csv")

Gravado: dados/processado/base_mensal.csv


## Caminho B: registros brutos (ou demonstração)

In [4]:
bruto = None
if agregado is None:
    bruto = carregar_bruto()
    MODO_DEMO = bruto is None
    if MODO_DEMO:
        bruto = gerar_registros_sinteticos()
        print(ROTULO_DEMO)
    print("Dimensões:", bruto.shape)
    print("Unidades (grafias brutas):", bruto[config.COL_UNIDADE].nunique())
    print("Datas ausentes:", int(pd.to_datetime(bruto[config.COL_DATA], errors="coerce").isna().sum()))
    if config.COL_ID in bruto.columns:
        print("Registros com id repetido:", int(bruto[config.COL_ID].duplicated().sum()))

In [5]:
if bruto is not None:
    limpo, log = limpar(bruto)
    display(log)
    mensal = agregar_mensal(limpo)
    resultado = validar_totais(limpo, mensal)
    print(resultado)
    assert resultado["totais_conferem"], "Total de registros limpos difere da soma mensal"
    nome = "base_mensal_DEMO.csv" if MODO_DEMO else "base_mensal.csv"
    mensal.to_csv(config.PROCESSADO / nome, index=False)
    print("Gravado: dados/processado/" + nome)